# OOP Triage Pipeline
This notebook demonstrates the OOP pipeline: preprocessors, feature builder, models, evaluator.

In [19]:
import sys, os
# Ensure project root is on path
proj_root = os.path.dirname(os.path.abspath("./"))
if proj_root not in sys.path:
    sys.path.append(proj_root)
from modules.data_loader import load_data
from modules.preprocessing import NumericalPreprocessor, CategoricalPreprocessor, TextPreprocessor, FeatureBuilder
from modules.models import TriagePipeline, LogisticRegressionModel
from modules.evaluation import ModelEvaluator

print('Imports OK')

Imports OK


In [20]:
# Load a small subset to keep runtime reasonable
cc, ph, train_df, test_df, sub = load_data()
print('Loaded: ', train_df.shape, test_df.shape, cc.shape, ph.shape)

Loaded:  (80000, 67) (20000, 64) (100000, 3) (100000, 26)


In [9]:
print('Train columns:', list(train_df.columns))
print('Has chief_complaint_raw:', 'chief_complaint_raw' in train_df.columns)

Train columns: ['patient_id', 'site_id', 'triage_nurse_id', 'arrival_mode', 'arrival_hour', 'arrival_day', 'arrival_month', 'arrival_season', 'shift', 'age', 'age_group', 'sex', 'language', 'insurance_type', 'transport_origin', 'pain_location', 'mental_status_triage', 'chief_complaint_system', 'num_prior_ed_visits_12m', 'num_prior_admissions_12m', 'num_active_medications', 'num_comorbidities', 'systolic_bp', 'diastolic_bp', 'mean_arterial_pressure', 'pulse_pressure', 'heart_rate', 'respiratory_rate', 'temperature_c', 'spo2', 'gcs_total', 'pain_score', 'weight_kg', 'height_cm', 'bmi', 'shock_index', 'news2_score', 'disposition', 'ed_los_hours', 'triage_acuity', 'chief_complaint_raw', 'hx_hypertension', 'hx_diabetes_type2', 'hx_diabetes_type1', 'hx_asthma', 'hx_copd', 'hx_heart_failure', 'hx_atrial_fibrillation', 'hx_ckd', 'hx_liver_disease', 'hx_malignancy', 'hx_obesity', 'hx_depression', 'hx_anxiety', 'hx_dementia', 'hx_epilepsy', 'hx_hypothyroidism', 'hx_hyperthyroidism', 'hx_hiv', 'h

In [6]:
print('CC columns:', list(cc.columns))
print('PH columns:', list(ph.columns))
print('Sample patient_id in train:', train_df['patient_id'].head())
print('Sample patient_id in cc:', cc['patient_id'].head())

CC columns: ['patient_id', 'chief_complaint_raw', 'chief_complaint_system']
PH columns: ['patient_id', 'hx_hypertension', 'hx_diabetes_type2', 'hx_diabetes_type1', 'hx_asthma', 'hx_copd', 'hx_heart_failure', 'hx_atrial_fibrillation', 'hx_ckd', 'hx_liver_disease', 'hx_malignancy', 'hx_obesity', 'hx_depression', 'hx_anxiety', 'hx_dementia', 'hx_epilepsy', 'hx_hypothyroidism', 'hx_hyperthyroidism', 'hx_hiv', 'hx_coagulopathy', 'hx_immunosuppressed', 'hx_pregnant', 'hx_substance_use_disorder', 'hx_coronary_artery_disease', 'hx_stroke_prior', 'hx_peripheral_vascular_disease']
Sample patient_id in train: 0    TG-UXRGA9UCO
1    TG-B19DBBS2G
2    TG-GZ97W7M6V
3    TG-THIB2TN9Q
4    TG-J3U3LQ2QY
Name: patient_id, dtype: str
Sample patient_id in cc: 0    TG-UXRGA9UCO
1    TG-B19DBBS2G
2    TG-GZ97W7M6V
3    TG-THIB2TN9Q
4    TG-J3U3LQ2QY
Name: patient_id, dtype: str


In [7]:
import importlib
import modules.data_loader
importlib.reload(modules.data_loader)
from modules.data_loader import load_data

In [21]:
# Instantiate preprocessors and feature builder
num = NumericalPreprocessor()
cat = CategoricalPreprocessor()
text = TextPreprocessor(text_col='chief_complaint_raw', max_features=500)
fb = FeatureBuilder([num, cat, text])
lr_model = LogisticRegressionModel()
pipeline = TriagePipeline([num, cat, text], fb, models=[lr_model])

# Fit preprocessors on training dataframe and build features
X = pipeline.run(train_df)
print('Feature matrix shape:', X.shape)

Feature matrix shape: (80000, 568)


In [22]:
# Split data
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, train_df['triage_acuity'], test_size=0.2, random_state=42, stratify=train_df['triage_acuity'])

# Train the model
lr_model.fit(X_train, y_train)

# Predict
y_pred = lr_model.predict(X_test)

# Evaluate
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           1       0.99      0.99      0.99       644
           2       1.00      0.99      0.99      2688
           3       1.00      1.00      1.00      5784
           4       1.00      1.00      1.00      4604
           5       1.00      1.00      1.00      2280

    accuracy                           1.00     16000
   macro avg       1.00      1.00      1.00     16000
weighted avg       1.00      1.00      1.00     16000



In [15]:
import importlib
import modules.oop_bases
importlib.reload(modules.oop_bases)
import modules.oop_models
importlib.reload(modules.oop_models)
from modules.oop_models import LogisticRegressionModel

In [13]:
# Quick model training (logistic) as demonstration
y = train_df['esi'] if 'esi' in train_df.columns else train_df.get('target')
if y is None:
    print('No target column found; skipping training')
else:
    lr = LogisticRegressionModel()
    lr.train(X.fillna(0), y)
    evaluator = ModelEvaluator()
    results = evaluator.evaluate({'LogisticRegression': lr}, X.fillna(0), y)
    print(results)

No target column found; skipping training
